# Norfolk County | Six shared-document questions


The [shared question document](https://docs.google.com/document/d/1Ba0FULReclAJJYfoqGe-NrArqX9EutvvRWoEKeXDgfs/edit?tab=t.0) ends with six questions for Norfolk. This notebook answers that section only.

**Coverage.** The source contains a 2017–2025 receipt cohort, or nine calendar years. The charts use that window and cannot supply a complete ten-year series. Trends by disposition year count dated outcomes in this intake cohort, so 2017 has little carry-in from earlier receipts and late cohorts can still resolve after 2025. Main outcome analyses use valid disposition dates even when receipt comes later. The chronology sensitivity restricts outcomes to dispositions on or after receipt. Receipt is a proxy for recorded intake volume where the question asks about charging. Missing stays separate from explicit unknown or other codes. Run all cells with pandas, matplotlib and IPython.

## Answers at a glance

| Question | What the Norfolk extract supports |
|---|---|
| 1. Disposition trends | Dismissals made up **67.4%** of dated DV-tagged outcome rows in 2017 and **70.3%** in 2025. For SA-tagged outcomes, the shares were **25.1%** and **26.4%**. CHAB had at most **10** eligible rows per year. The explicit child-harm charge proxy has a separate series. |
| 2. District court outcomes | Across the five district courts, the share of case-court pairs with any recorded dismissal ranged from **65.9% in Dedham** to **80.5% in Stoughton**. Case mix and missing court coverage limit comparison. |
| 3. Crime mix | **MMVE** made up **32.1%** of countywide charge rows. Agency-level mixes are available. Incident municipality is absent. |
| 4. Victims by race or ethnicity | Unavailable. Victim attributes and linkage keys are absent. |
| 5. Charging patterns | DV-tagged received cases moved from **1,726 to 1,540** between 2017 and 2025, and SA-tagged cases from **169 to 212**. Child and elder indicators are definition-sensitive. Hate-crime trends require a validated category. |
| 6. Nolle prosequi | **3,545 District** and **352 Superior** case IDs had at least one recorded NOPR charge in the window. **NARC** was the most common crime code by distinct case IDs in each court level. The notebook also shows missing court and a stricter all-observed-charges measure. |

Each figure has a corresponding CSV table under the notebook's output folder. The six results use nine years of available records and distinguish case counts from charge-row percentages.

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

filename = 'NFK-Data-final-2017-2025 redacted.csv'
source = Path(filename) if Path(filename).is_file() else Path('norfolk') / filename
sha256 = hashlib.sha256(source.read_bytes()).hexdigest()
raw = pd.read_csv(source, encoding='cp1252', dtype=str, keep_default_na=False)
d = raw.apply(lambda s: s.str.strip().replace('', 'Missing'))
d['record'] = d.index + 1  # One-based data-record ordinal after header.
YEARS = list(range(2017, 2026))
for field, alias, fmt in [
    ('Received Date','received','%m/%d/%Y'),
    ('Date Crm','offense','%Y-%m-%d %I:%M%p'),
    ('Date Dspstn','disposed','%Y-%m-%d %I:%M%p'),
    ('Date Exctn','execution','%Y-%m-%d %I:%M%p')
]:
    parsed = pd.to_datetime(d[field], format=fmt, errors='coerce')
    valid = parsed.between('1900-01-01','2026-11-06')
    d[alias] = parsed.where(valid)
    d[alias+'_quality'] = 'Valid'
    d.loc[d[field].eq('Missing'), alias+'_quality'] = 'Missing'
    d.loc[d[field].ne('Missing') & ~valid, alias+'_quality'] = 'Invalid / out of bounds'

d['received_year'] = d.received.dt.year.astype('Int64')
d['disposed_year'] = d.disposed.dt.year.astype('Int64')
d['disposition'] = d['Desc Dspstn']
cwof = d['Code Dspstn'].eq('PLEA') & d['Code Rsn Dspstn'].eq('CWOF')
d.loc[cwof,'disposition'] = 'CWOF (plea reason)'
d.loc[d['Code Dspstn'].eq('PLEA') & ~cwof,'disposition'] = 'Other recorded plea'
d['days_offense_received'] = (d.received - d.offense).dt.days
d['days_received_disposition'] = (d.disposed - d.received).dt.days
d['days_offense_disposition'] = (d.disposed - d.offense).dt.days
court_names = {'QDC':'Quincy', 'BDC':'Brookline', 'DDC':'Dedham',
               'SDC':'Stoughton','WDC':'Wrentham','NSC':'Norfolk Superior'}
d['court'] = d['Code Crt Lctn Disp'].map(court_names).fillna(d['Code Crt Lctn Disp'])
d['court_level'] = 'Other / unmapped'
d.loc[d.court.isin(list(court_names.values())[:5]),'court_level'] = 'District'
d.loc[d.court.eq('Norfolk Superior'),'court_level'] = 'Superior'
d.loc[d.court.eq('Missing'),'court_level'] = 'Missing'
window = d[d.disposed_year.isin(YEARS) & d['Code Dspstn'].ne('Missing')].copy()
chronological = window[window.days_received_disposition.ge(0)].copy()
cohort = d[d.received_year.isin(YEARS)].copy()
plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':110})
palette = ['#27647b','#dc8951','#719b81','#845e91','#b7575c','#8a8f98',
           '#a9b9c5','#9e7948','#434979','#72714b','#a87caa','#456a52']
def chart(data, title, ylabel, kind='bar', stacked=False, xlabel='', figsize=(10,4)):
    ax = data.plot(kind=kind, stacked=stacked, figsize=figsize, color=palette)
    ax.set(title=title, ylabel=ylabel, xlabel=xlabel)
    if kind == 'bar':
        ax.tick_params(axis='x', labelrotation=0)
    if ax.get_legend():
        ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', title=None, fontsize=8)
    plt.tight_layout()
    plt.show()
    plt.close(ax.figure)
def table(frame, name, limit=16):
    frame.to_csv(out / (name + '.csv'))
    display(frame.head(limit).round(2))
def mix(frame, by, value, name, title, top=7):
    frequent = frame[value].value_counts().head(top).index.union(['Missing'], sort=False)
    shown = frame[value].where(frame[value].isin(frequent), 'Other recorded')
    counts = pd.crosstab(frame[by], shown)
    pct = counts.div(counts.sum(axis=1),axis=0).mul(100)
    table(pd.crosstab(frame[by],frame[value]),name+'_full_counts')
    table(pct,name+'_chart_pct')
    chart(pct,title,'Percent of rows',stacked=True)
def summary_days(x):
    usable = x[x.ge(0)]
    return pd.Series({'all_rows':len(x),'missing_or_invalid':x.isna().sum(),
                      'negative':x.lt(0).sum(),'usable':len(usable),
                      'mean':usable.mean(),'p25':usable.quantile(.25),
                      'median':usable.median(),'p75':usable.quantile(.75),
                      'p95':usable.quantile(.95),'max':usable.max()})

out = source.parent / 'norfolk_questions_outputs'
out.mkdir(exist_ok=True)

## Definitions used across questions
DV and SA are source **case-type** tags, so their series include accompanying charges. CHAB, ELAB and HACR are **crime-type** codes whose meanings need codebook confirmation. A second series uses named charge codes or descriptions as narrower checks. The cohorts can overlap and must not be added together.

In [ ]:
desc = d['Desc Chrg'].str.upper()
charge = d['Code Chrg']
case_type = d['Case Type']
crime = d['Crime Type']
groups = {
    'DV case tag': case_type.eq('DV'),
    'SA case tag': case_type.eq('SA'),
    'CHAB crime code': crime.eq('CHAB'),
    'ELAB crime code': crime.eq('ELAB'),
    'HACR crime code (unverified)': crime.eq('HACR'),
    'Family / household charge': desc.str.contains(r'FAMILY\s*/\s*HOUSEHOLD'),
    'RAPE or INDO crime code': crime.isin(['RAPE','INDO']),
    'Explicit child harm charge proxy': charge.str.startswith(('265/13J','265/13L','119/39')),
    'Elder / disabled charge proxy': charge.str.startswith('265/13K'),
    'Civil-rights charge proxy': charge.str.startswith(('265/37','12/11J'))
}
audit = []
for group, mask in groups.items():
    g = d.loc[mask]
    display(Markdown(f'**{group}** — {len(g):,} rows in {g["Id Cs"].nunique():,} distinct cases.'))
    for (code, description), n in g.groupby(['Code Chrg','Desc Chrg']).size().items():
        audit.append((group,code,description,n))
table(pd.DataFrame(audit,columns=['definition','charge_code','description','rows'])
      .sort_values(['definition','rows'],ascending=[True,False]),'category_definition_audit',10)
display(d.loc[groups['HACR crime code (unverified)'],'Desc Chrg'].value_counts().to_frame('rows'))

## 1. How have dispositions for domestic violence, sexual assault, and child abuse changed?

**Partially answerable.** Compare recorded charge outcomes within the DV and SA case tags and the CHAB crime code. A CHAB row is a source coding signal, and an explicit child-harm charge proxy provides a definition check. The outcome denominator is qualifying charge rows within each group and disposition year. All outcome categories are exported. The charts pool rare labels under Other recorded and display annual denominators separately. Small CHAB denominators can move the percentages substantially.

In [ ]:
for name in ['DV case tag','SA case tag','CHAB crime code','Explicit child harm charge proxy']:
    selected = window.loc[groups[name].reindex(window.index)]
    key = name.split()[0].lower()
    counts = pd.crosstab(selected.disposed_year,selected.disposition).reindex(YEARS,fill_value=0)
    table(counts,'q1_'+key+'_outcome_counts',10)
    n = counts.sum(axis=1)
    leading = list(counts.sum().nlargest(6).index)
    shown = counts[leading].copy()
    shown['Other recorded'] = n - shown.sum(axis=1)
    pct = shown.div(n.where(n.gt(0)),axis=0).mul(100)
    table(pct,'q1_'+key+'_outcome_pct',10)
    chart(pct,f'{name} | outcomes by disposition year','Percent of outcome rows',stacked=True)
    comparison = n.to_frame('eligible_rows')
    # Keep the same outcome labels for the chronology sensitivity.
    timed = selected[selected.days_received_disposition.ge(0)]
    timed_counts = pd.crosstab(timed.disposed_year,timed.disposition).reindex(
        index=YEARS,columns=counts.columns,fill_value=0)
    comparison['rows_disposition_on_or_after_receipt'] = timed_counts.sum(axis=1)
    table(timed_counts,'q1_'+key+'_chronology_sensitivity_counts',10)
    if 'Dismissed' in counts and n.loc[2017] and n.loc[2025]:
        print(f"Dismissed share: {counts.at[2017,'Dismissed']/n.loc[2017]:.1%} in 2017 and "
              f"{counts.at[2025,'Dismissed']/n.loc[2025]:.1%} in 2025.")
    print(f"Annual outcome denominators range from {n.min():,} to {n.max():,} rows.")
    table(comparison,'q1_'+key+'_denominators')

## 2. How do outcomes vary across Quincy, Brookline, Dedham, Stoughton and Wrentham?

**Answerable descriptively for the recorded cohort.** Use the recorded disposition location and inferred abbreviation mapping. Show charge outcomes, plus the share of case-court pairs with each selected outcome. A pair may have multiple outcomes and a case may appear at multiple courts. Rates are unadjusted for case mix. Missing court stays visible and court may be recorded differently across charges in the same case.

In [ ]:
court_counts = pd.crosstab(window.court,window.disposition)
table(court_counts,'q2_charge_outcome_counts')
table(pd.crosstab(chronological.court,chronological.disposition),'q2_chronology_sensitivity_counts')
named = window[window.court_level.eq('District')]
mix(named,'court','disposition','q2_district','Charge outcome composition across five district courts')
pairs = window[['Id Cs','court','disposition']].drop_duplicates()
den = pairs.groupby('court')['Id Cs'].nunique()
case_counts = pd.crosstab(pairs.court,pairs.disposition)
case_rates = case_counts.div(den,axis=0).mul(100)
table(case_rates.join(den.rename('case_court_pairs')),'q2_case_court_any_outcome_pct')
primary = ['Dismissed','Nol Pros','CWOF (plea reason)','Other recorded plea','Guilty','Not Guilty']
district_rates = case_rates.reindex(list(court_names.values())[:5])
dismissed = district_rates['Dismissed']
print(f"Any dismissal ranged from {dismissed.min():.1f}% in {dismissed.idxmin()} "
      f"to {dismissed.max():.1f}% in {dismissed.idxmax()}.")
chart(district_rates[primary],
      'Case-court pairs with selected outcomes','Percent of case-court pairs',figsize=(10,5))

## 3. What is the breakdown of crime types by municipality and countywide?

**County extract answerable. Municipality unavailable.** The county table measures the composition of recorded charge rows by source crime-type code. The agency table gives a separate administrative breakdown. Police agencies can handle incidents across borders, and State Police and other agencies cannot be assigned to incident municipalities from these fields. This extract measures recorded prosecution activity rather than all incidents.

In [ ]:
county = cohort['Crime Type'].value_counts().rename('charge_rows').to_frame()
county['percent_of_charge_rows'] = county.charge_rows / len(cohort) * 100
table(county,'q3_county_crime_mix',20)
chart(county.percent_of_charge_rows.head(15).sort_values(),'Most frequent countywide crime codes',
      'Crime type code',kind='barh',figsize=(9,5))
agency_counts = pd.crosstab(cohort['Code Ofcr Agncy'],cohort['Crime Type'])
agency_pct = agency_counts.div(agency_counts.sum(axis=1),axis=0).mul(100)
table(agency_counts,'q3_agency_crime_counts',6)
table(agency_pct.join(agency_counts.sum(axis=1).rename('all_agency_rows')),
      'q3_agency_crime_pct',6)
top_agencies = agency_counts.sum(axis=1).drop('Missing').nlargest(8).index.union(['Missing'],sort=False)
mix(cohort[cohort['Code Ofcr Agncy'].isin(top_agencies)],'Code Ofcr Agncy','Crime Type',
    'q3_top_agencies','Crime mix of highest-volume agencies and Missing',6)

## 4. What is the breakdown of victims by race/ethnicity, overall and over time?

**Unavailable.** The CSV has no victim ID, victim role, or victim race/ethnicity field. The person race code appears on the prosecution record. Victim-level trends would need a victim-level dataset with time coverage and a victim linkage key.

## 5. How have charging patterns changed for DV, sexual assault, child abuse, elder abuse, and hate crimes?

**Partially answerable.** Compare annual case counts, charge rows, and shares of all recorded receipts. This is the observed charge inventory grouped by receipt year. There is no original-charge history or confirmed filing date.

The DV/SA case tags are broad. CHAB and ELAB are provisional code indicators and include a range of charge descriptions. The elder charge proxy combines elder and disabled wording, so elder-only totals remain unavailable. HACR occurs alongside a habitual-offender charge description. That could be an enhancement within a flagged matter, but the code meaning and bias motivation are unverified. Civil-rights charges are a separate observable proxy. A defensible hate-crime trend requires a verified definition and code mapping.

In [ ]:
base = cohort.groupby('received_year').agg(all_rows=('record','size'),all_cases=('Id Cs','nunique'))
pieces = []
for name, mask in groups.items():
    sub = cohort.loc[mask.reindex(cohort.index)]
    a = sub.groupby('received_year').agg(charge_rows=('record','size'),cases=('Id Cs','nunique'))
    a = a.reindex(YEARS,fill_value=0).join(base)
    a['charge_share_pct'] = a.charge_rows / a.all_rows * 100
    a['case_share_pct'] = a.cases / a.all_cases * 100
    pieces.append(a.assign(group=name))
annual_groups = pd.concat(pieces).rename_axis('year').reset_index()
table(annual_groups.set_index(['group','year']),'q5_annual_receipt_cohorts',18)
p = annual_groups.pivot(index='year',columns='group',values='cases')
table(p,'q5_case_counts_for_all_definitions',10)
for indicator in ['DV case tag','SA case tag','CHAB crime code','ELAB crime code',
                  'Explicit child harm charge proxy','Elder / disabled charge proxy']:
    print(f"{indicator}: {p.at[2017,indicator]:,} cases in 2017 and {p.at[2025,indicator]:,} in 2025.")
shares = annual_groups.pivot(index='year',columns='group',values='case_share_pct')
chart(shares[['DV case tag','SA case tag','Family / household charge','RAPE or INDO crime code']],
      'DV and SA indicators by receipt year','Percent of received cases',kind='line')
chart(p[['CHAB crime code','Explicit child harm charge proxy','ELAB crime code',
         'Elder / disabled charge proxy','Civil-rights charge proxy','HACR crime code (unverified)']],
      'Less frequent indicators by receipt year','Distinct cases',kind='line',figsize=(9,4))

## 6. What is the nolle prosequi case breakdown by crime type in District and Superior Court?

**Answerable for the observed window.** Use the source code NOPR, labeled Nol Pros. A case qualifies in a court level if at least one eligible charge in that level received NOPR. Assign crime types from those NOPR charges. The same case can appear under several crime types or both court levels, so case percentages across types may sum above 100%. Charge composition has a separate denominator and sums to 100% within each level. Missing and unmapped courts stay separate.

The all-observed-outcomes sensitivity counts case-level pairs whose eligible charges all received NOPR. Restrict that calculation to source DISP cases without a recorded post-window disposition or outcome-bearing rows outside the eligible set, and require all recorded case charge rows to be eligible and in the same level. Even this stricter rule is limited to observed rows.

In [ ]:
nop = window[window['Code Dspstn'].eq('NOPR')]
level = nop.groupby('court_level').agg(nopr_charge_rows=('record','size'),nopr_cases=('Id Cs','nunique'))
type_mix = nop.groupby(['court_level','Crime Type']).agg(nopr_charge_rows=('record','size'),
                                                           nopr_cases=('Id Cs','nunique'))
type_mix = type_mix.join(level.add_prefix('level_'),on='court_level')
type_mix['charge_pct'] = type_mix.nopr_charge_rows / type_mix.level_nopr_charge_rows * 100
type_mix['case_pct_overlapping_types'] = type_mix.nopr_cases / type_mix.level_nopr_cases * 100
table(level,'q6_totals_by_level')
table(nop[nop.days_received_disposition.ge(0)].groupby('court_level').agg(
    nopr_charge_rows=('record','size'),nopr_cases=('Id Cs','nunique')),
    'q6_chronology_sensitivity_totals')
for level_name in ['District','Superior']:
    highest = type_mix.loc[level_name].sort_values('nopr_cases',ascending=False).iloc[0]
    print(f"{level_name}: {level.at[level_name,'nopr_cases']:,} cases with any NOPR. "
          f"Leading crime code {highest.name}, {highest.nopr_cases:,.0f} cases "
          f"({highest.case_pct_overlapping_types:.1f}%).")
table(type_mix.sort_values(['court_level','nopr_cases'],ascending=[True,False]),
      'q6_crime_type_breakdown',22)
table(nop.groupby(['disposed_year','court_level','Crime Type']).agg(
      rows=('record','size'),cases=('Id Cs','nunique')),'q6_year_level_type')
level_year = nop.groupby(['disposed_year','court_level'])['Id Cs'].nunique().unstack(fill_value=0)
table(level_year,'q6_distinct_cases_by_year_and_level')
chart(level_year.reindex(YEARS,fill_value=0),'Cases with any NOPR by disposition year','Distinct cases',kind='line')
leading = nop[nop.court_level.isin(['District','Superior'])]['Crime Type'].value_counts().head(12).index
shown = type_mix.charge_pct.unstack('court_level').reindex(leading)[['District','Superior']].fillna(0)
chart(shown.iloc[::-1],'Charge composition within NOPR outcomes by court level',
      'Crime type',kind='barh',figsize=(9,5))
all_raw = d.groupby('Id Cs').size()
strict = nop.groupby(['Id Cs','court_level']).agg(rows=('record','size'),status=('Case Status','first'))
strict = strict[strict.status.eq('DISP') & strict.rows.eq(strict.index.get_level_values('Id Cs').map(all_raw))]
table(strict.groupby('court_level').size().rename('cases_all_source_rows_eligible_nopr').to_frame(),
      'q6_all_observed_nopr_sensitivity')

In [ ]:
assert hashlib.sha256(source.read_bytes()).hexdigest() == sha256
print('Source SHA-256:', sha256)
print('Aggregate CSV outputs:', len(list(out.glob('*.csv'))))